# **Data Loading**

In [1]:
# Import the required libraries
import pandas as pd
import numpy as np

# Load the dataset
df = pd.read_csv("titanic_train.csv")

# Show the dataset
print(df.head())

   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   

   Parch            Ticket     Fare Cabin Embarked  
0      0         A/5 21171   7.2500   NaN        S  
1      0          PC 17599  71.2833   C85        C  
2      0  STON/O2. 3101282   7.9250   NaN        S  
3      0            113803  53.1000  C123        S  
4      0            373450   8.0500   NaN        S  


# **Data Exploration**

In [2]:
print("Shape:", df.shape)

print("\nUnique values:")
print(df.nunique())

print("\nData types:")
print(df.dtypes)

print("\nSummary statistics:")
print(df.describe)

print("\nTarget balance:")
print(df["Survived"].value_counts())

print("\nMissing values:")
print(df.isnull().sum()[df.isnull().sum() > 0])

print("\nDuplicate values:")
print(df.duplicated().sum())

Shape: (891, 12)

Unique values:
PassengerId    891
Survived         2
Pclass           3
Name           891
Sex              2
Age             88
SibSp            7
Parch            7
Ticket         681
Fare           248
Cabin          147
Embarked         3
dtype: int64

Data types:
PassengerId      int64
Survived         int64
Pclass           int64
Name            object
Sex             object
Age            float64
SibSp            int64
Parch            int64
Ticket          object
Fare           float64
Cabin           object
Embarked        object
dtype: object

Summary statistics:
<bound method NDFrame.describe of      PassengerId  Survived  Pclass  \
0              1         0       3   
1              2         1       1   
2              3         1       3   
3              4         1       1   
4              5         0       3   
..           ...       ...     ...   
886          887         0       2   
887          888         1       1   
888          889         0

# **Manual Approcah**

In [3]:
# Import the required libraires
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# Cross validation
y = df["Survived"]
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# --- Manual: fill, dummy, scale, all by hand ---
m = df.copy()
m["Age"] = m["Age"].fillna(m["Age"].median())
m["Embarked"] = m["Embarked"].fillna(m["Embarked"].mode()[0])
m = m.drop(columns=["PassengerId", "Name", "Ticket", "Cabin"])
m = pd.get_dummies(m, columns=["Sex", "Embarked", "Pclass"], drop_first=True)

X_manual = m.drop(columns=["Survived"])
X_manual_scaled = StandardScaler().fit_transform(X_manual)

manual_acc = cross_val_score(LogisticRegression(max_iter=1000), X_manual_scaled, y, cv=cv, scoring="accuracy")
manual_auc = cross_val_score(LogisticRegression(max_iter=1000), X_manual_scaled, y, cv=cv, scoring="roc_auc")

print(f"MANUAL  accuracy: {manual_acc.mean():.4f}   roc_auc: {manual_auc.mean():.4f}")

MANUAL  accuracy: 0.7969   roc_auc: 0.8511


# **Pipeline With Column Transformer**

In [4]:
# Import the required libraries
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

# Configuring the columns
X = df.drop(columns=["Survived", "PassengerId", "Name", "Ticket", "Cabin"])

NUMERICAL   = ["Age", "SibSp", "Parch", "Fare"]
CATEGORICAL = ["Pclass", "Sex", "Embarked"]

# Each branch is itself a mini-pipeline: impute first, then transform
numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler",  StandardScaler()),
])

categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot",  OneHotEncoder(handle_unknown="ignore", drop="first")),
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipe,     NUMERICAL),
    ("cat", categorical_pipe, CATEGORICAL),
])

pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=1000, random_state=42)),
])

acc = cross_val_score(pipe, X, y, cv=cv, scoring="accuracy")
auc = cross_val_score(pipe, X, y, cv=cv, scoring="roc_auc")
print(f"PIPELINE  accuracy: {acc.mean():.4f}   roc_auc: {auc.mean():.4f}")

PIPELINE  accuracy: 0.8036   roc_auc: 0.8511


# **Feature Engineering**

In [5]:
def engineer_features(data):
    d = data.copy()

    # 1. Family size (siblings + parents/children + self)
    d["FamilySize"] = d["SibSp"] + d["Parch"] + 1

    # 2. Travelling alone
    d["IsAlone"] = (d["FamilySize"] == 1).astype(int)

    # 3. Title extracted from Name
    t = d["Name"].str.extract(r",\s*([^\.]+)\.", expand=False).str.strip()
    t = t.replace({"Mlle": "Miss", "Ms": "Miss", "Mme": "Mrs"})
    d["Title"] = t.where(t.isin(["Mr", "Miss", "Mrs", "Master"]), "Rare")

    # 4. Cabin recorded at all (Cabin is 77% missing — but the missingness itself may carry signal)
    d["HasCabin"] = d["Cabin"].notna().astype(int)

    return d

df_eng = engineer_features(df)
print(df_eng["Title"].value_counts())

Title
Mr        517
Miss      185
Mrs       126
Master     40
Rare       23
Name: count, dtype: int64


# **Pipeline With Engineered Features**

In [6]:
X_eng = df_eng.drop(columns=["Survived", "PassengerId", "Name", "Ticket", "Cabin"])

NUMERICAL_E   = ["Age", "SibSp", "Parch", "Fare", "FamilySize"]
CATEGORICAL_E = ["Pclass", "Sex", "Embarked", "Title"]
PASSTHROUGH   = ["IsAlone", "HasCabin"]

preprocessor_eng = ColumnTransformer([
    ("num",  numeric_pipe,     NUMERICAL_E),
    ("cat",  categorical_pipe, CATEGORICAL_E),
    ("pass", "passthrough",    PASSTHROUGH),
])

pipe_eng = Pipeline([
    ("preprocessor", preprocessor_eng),
    ("model", LogisticRegression(max_iter=1000, random_state=42)),
])

acc_e = cross_val_score(pipe_eng, X_eng, y, cv=cv, scoring="accuracy")
auc_e = cross_val_score(pipe_eng, X_eng, y, cv=cv, scoring="roc_auc")

print(f"PIPELINE + ENGINEERED  accuracy: {acc_e.mean():.4f}   roc_auc: {auc_e.mean():.4f}")
print(f"Improvement over plain pipeline: {acc_e.mean()-acc.mean():+.4f} accuracy, {auc_e.mean()-auc.mean():+.4f} AUC")

PIPELINE + ENGINEERED  accuracy: 0.8294   roc_auc: 0.8718
Improvement over plain pipeline: +0.0258 accuracy, +0.0207 AUC


# **Testing Features Which Helped**

In [7]:
def score_config(num_cols, cat_cols, pass_cols):
    steps = [("num", numeric_pipe, num_cols), ("cat", categorical_pipe, cat_cols)]
    if pass_cols:
        steps.append(("pass", "passthrough", pass_cols))
    p = Pipeline([("preprocessor", ColumnTransformer(steps)),
                  ("model", LogisticRegression(max_iter=1000, random_state=42))])
    return (cross_val_score(p, X_eng, y, cv=cv, scoring="accuracy").mean(),
            cross_val_score(p, X_eng, y, cv=cv, scoring="roc_auc").mean())

configs = {
    "baseline (no engineering)": (["Age","SibSp","Parch","Fare"], ["Pclass","Sex","Embarked"], []),
    "+ FamilySize & IsAlone":    (["Age","SibSp","Parch","Fare","FamilySize"], ["Pclass","Sex","Embarked"], ["IsAlone"]),
    "+ Title only":              (["Age","SibSp","Parch","Fare"], ["Pclass","Sex","Embarked","Title"], []),
    "+ HasCabin only":           (["Age","SibSp","Parch","Fare"], ["Pclass","Sex","Embarked"], ["HasCabin"]),
    "ALL engineered":            (NUMERICAL_E, CATEGORICAL_E, PASSTHROUGH),
}

for name, (n, c, p) in configs.items():
    a, u = score_config(n, c, p)
    print(f"{name:28s} acc={a:.4f}  auc={u:.4f}")

baseline (no engineering)    acc=0.8036  auc=0.8511
+ FamilySize & IsAlone       acc=0.8014  auc=0.8558
+ Title only                 acc=0.8227  auc=0.8694
+ HasCabin only              acc=0.8036  auc=0.8554
ALL engineered               acc=0.8294  auc=0.8718


# **Final Holdout Evaluation**

In [8]:
# Import the required libraries
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report, confusion_matrix

# Train-test split
X_tr, X_te, y_tr, y_te = train_test_split(X_eng, y, test_size=0.2, stratify=y, random_state=42)

# Evaluation
pipe_eng.fit(X_tr, y_tr)
pred = pipe_eng.predict(X_te)
prob = pipe_eng.predict_proba(X_te)[:, 1]

print(f"Holdout accuracy: {accuracy_score(y_te, pred):.4f}")
print(f"Holdout ROC-AUC : {roc_auc_score(y_te, prob):.4f}")
print()
print(confusion_matrix(y_te, pred))
print(classification_report(y_te, pred, target_names=["Died", "Survived"]))

Holdout accuracy: 0.8156
Holdout ROC-AUC : 0.8676

[[97 13]
 [20 49]]
              precision    recall  f1-score   support

        Died       0.83      0.88      0.85       110
    Survived       0.79      0.71      0.75        69

    accuracy                           0.82       179
   macro avg       0.81      0.80      0.80       179
weighted avg       0.81      0.82      0.81       179



# **Pipeline Saving And Verification**

In [9]:
# Import the required libraries
import joblib

# Refit on ALL the data before saving — you want the deployed model to use everything
pipe_eng.fit(X_eng, y)
joblib.dump(pipe_eng, "titanic_pipeline.joblib")
print("Saved.")

# --- Verification: never trust a save you haven't loaded back ---
loaded = joblib.load("titanic_pipeline.joblib")

original_preds = pipe_eng.predict_proba(X_eng)[:, 1]
loaded_preds   = loaded.predict_proba(X_eng)[:, 1]
print("Predictions identical after reload:", np.allclose(original_preds, loaded_preds))

# The whole point of a pipeline: it handles raw, messy input end-to-end
messy = X_eng.iloc[[0]].copy()
messy["Age"] = np.nan
messy["Embarked"] = np.nan
print("Handles raw input with NaN:", loaded.predict(messy))

Saved.
Predictions identical after reload: True
Handles raw input with NaN: [0]
